In [1]:
import io
import boto3
import pandas as pd
from tqdm import tqdm
from typing import List
import sys
sys.path.append("/home/sagemaker-user/price-elasticity/scripts")
from aws import list_s3_objects, load_file_from_s3

s3_client = boto3.client("s3")

def load_multiple_parquets_from_s3(
    s3client,
    file_keys: List[str],
    bucket: str
) -> pd.DataFrame:
    """
    Loads and combines multiple parquet files from S3 given a list of file keys.

    Parameters:
    - s3client: Boto3 S3 client object
    - file_keys (List[str]): List of S3 file keys (paths) to load
    - bucket (str): The name of the S3 bucket

    Returns:
    - pandas.DataFrame: A combined DataFrame containing data from all parquet files

    Example:
    >>> s3 = boto3.client('s3')
    >>> files = ['data/part-00000.parquet', 'data/part-00001.parquet']
    >>> df = load_multiple_parquets_from_s3(s3, files, "my-bucket")
    """
    if not file_keys:
        raise ValueError("file_keys list is empty")

    print(f"Loading {len(file_keys)} parquet file(s)")

    dfs = []
    for i, file_key in enumerate(file_keys, 1):
        print(f"Loading {i}/{len(file_keys)}: {file_key.split('/')[-1]}")
        obj = s3client.get_object(Bucket=bucket, Key=file_key)
        df = pd.read_parquet(io.BytesIO(obj['Body'].read()))
        dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True)
    print(f"Combined DataFrame shape: {combined_df.shape}")

    return combined_df


def upload_to_s3(df, bucket_name, s3_key):
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False)
    buffer.seek(0)
    s3_client.upload_fileobj(buffer, bucket_name, s3_key)

SALES_BUCKET = "prime-rel-hoogvliet"
SALES_FOLDER = "prime/platform/solutions/price_elasticity/inputs/total/"
COMPETITOR_PATH = "/home/sagemaker-user/price-elasticity/scripts/competitor_prices (1).parquet"

OUT_BUCKET = "prime-rel-hoogvliet"
OUT_FOLDER = "prime/platform/temp/daily_sales/sales_with_competitors/"

print("Loading data...")
comp_df = pd.read_parquet(COMPETITOR_PATH)
comp_df["price_date"] = pd.to_datetime(comp_df["price_date"], errors="coerce")
comp_df["product_code"] = comp_df["product_code"].astype(str)

# Handling possible duplicates in competitor data by taking the median price for the same product, date and competitor
comp_grouped = comp_df.groupby(
    ["product_code", "price_date", "competitor_name"], as_index=False
)["normalized_price"].median()

print("Pivoting competitor data...")
# If a competitor has no price for a date, Pivot will leave it as NaN (Null).
competitor_pivot = comp_grouped.pivot(
    index=["product_code", "price_date"],
    columns="competitor_name",
    values="normalized_price"
).reset_index()

# Clean up column names for the final output
competitor_pivot.columns.name = None
competitor_pivot = competitor_pivot.rename(columns={"price_date": "join_date"})

# Rename competitor columns to: jumbo_price, albert_heijn_price, etc.
competitor_pivot = competitor_pivot.rename(
    columns={
        c: f"{str(c).lower().strip().replace(' ', '_')}_price"
        for c in competitor_pivot.columns
        if c not in ["product_code", "join_date"]
    }
)

sales_keys = [k for k in list_s3_objects(s3_client, SALES_BUCKET, SALES_FOLDER) if k.endswith(".parquet")]
sales_keys = sorted(sales_keys)

sales_df = load_multiple_parquets_from_s3(s3_client, sales_keys, SALES_BUCKET)

sales_df["join_date"] = pd.to_datetime(
    sales_df["date_code"].astype(str), format="%Y%m%d", errors="coerce"
)

# Ensure both product_codes are strings so the merge works
sales_df["product_code"] = sales_df["product_code"].astype(str)

# 'left' join keeps all sales rows and inserts Nulls where competitor prices are missing
final_df = sales_df.merge(
    competitor_pivot,
    on=["product_code", "join_date"],
    how="left"
).drop(columns=["join_date"])

final_df["year_week"] = final_df["yearweek"].astype(str)

Loading data...
Pivoting competitor data...
Loading 13776 parquet file(s)
Loading 1/13776: part-00000-a8a3b3c1-6f6f-49fc-9b2c-58ac3d3ffc88.c000.snappy.parquet
Loading 2/13776: part-00001-a8a3b3c1-6f6f-49fc-9b2c-58ac3d3ffc88.c000.snappy.parquet
Loading 3/13776: part-00002-a8a3b3c1-6f6f-49fc-9b2c-58ac3d3ffc88.c000.snappy.parquet
Loading 4/13776: part-00003-a8a3b3c1-6f6f-49fc-9b2c-58ac3d3ffc88.c000.snappy.parquet
Loading 5/13776: part-00004-a8a3b3c1-6f6f-49fc-9b2c-58ac3d3ffc88.c000.snappy.parquet
Loading 6/13776: part-00005-a8a3b3c1-6f6f-49fc-9b2c-58ac3d3ffc88.c000.snappy.parquet
Loading 7/13776: part-00006-a8a3b3c1-6f6f-49fc-9b2c-58ac3d3ffc88.c000.snappy.parquet
Loading 8/13776: part-00007-a8a3b3c1-6f6f-49fc-9b2c-58ac3d3ffc88.c000.snappy.parquet
Loading 9/13776: part-00008-a8a3b3c1-6f6f-49fc-9b2c-58ac3d3ffc88.c000.snappy.parquet
Loading 10/13776: part-00009-a8a3b3c1-6f6f-49fc-9b2c-58ac3d3ffc88.c000.snappy.parquet
Loading 11/13776: part-00010-a8a3b3c1-6f6f-49fc-9b2c-58ac3d3ffc88.c000.snap

In [2]:
CHUNK_SIZE = 2_000_000

for yw, part_df in final_df.groupby("year_week", sort=True):
    part_df = part_df.reset_index(drop=True)
    num_parts = (len(part_df) + CHUNK_SIZE - 1) // CHUNK_SIZE

    s = str(yw)
    yw_fmt = f"{s[:4]}_{s[4:6]}" if s.isdigit() and len(s) == 6 else s.replace("-", "_")

    for part_idx in range(num_parts):
        start = part_idx * CHUNK_SIZE
        end = min((part_idx + 1) * CHUNK_SIZE, len(part_df))
        out_key = f"{OUT_FOLDER}year_week={yw_fmt}/part-{part_idx:05d}.parquet"
        upload_to_s3(part_df.iloc[start:end], OUT_BUCKET, out_key)

print("DONE")

(15416307, 20)

In [3]:
comp_df.shape

(26494421, 12)

In [4]:
final_df.shape

(15416307, 27)

In [5]:
sales_df.to_parquet("sales_test.parquet", index=False)
final_df.to_parquet("final_test.parquet", index=False)

import os
print("sales_test MB:", os.path.getsize("sales_test.parquet")/1024**2)
print("final_test MB:", os.path.getsize("final_test.parquet")/1024**2)

sales_test MB: 364.16014099121094
final_test MB: 366.95208263397217


In [6]:
print("sales_df memory MB:", sales_df.memory_usage(deep=True).sum()/1024**2)
print("final_df memory MB:", final_df.memory_usage(deep=True).sum()/1024**2)

sales_df memory MB: 8097.426712989807
final_df memory MB: 9611.746725082397
